## original fertility test

In [7]:
import random
import pandas as pd
from pathlib import Path
from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.pre_tokenizers import Whitespace

# Load tokenizer
TOKENIZER_PATH = Path("../shared_tokenizer2/tokenizer.json").resolve()
assert TOKENIZER_PATH.exists(), f"Tokenizer not found: {TOKENIZER_PATH}"

tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))
whitespace = Whitespace()

DATASETS = {
    "English": "BabyLM-community/babylm-eng",
    "Dutch": "BabyLM-community/babylm-nld",
    "Indonesian": "BabyLM-community/babylm-ind",
    "Javanese": "BabyLM-community/babylm-jav",
}

MAX_WORDS_PER_LANG = 300_000

# 每个 chunk 的大小
CHUNK_WORDS = 5_000
SEED = 42


def count_tokens(text):
    # 重要：分 chunk 算的时候，不要给每个 chunk 加 special tokens
    return len(tokenizer.encode(text, add_special_tokens=False).ids)


def get_text_column(dataset):
    candidates = ["text", "sentence", "content", "raw_text"]
    for col in candidates:
        if col in dataset.column_names:
            return col
    raise ValueError(f"Cannot find text column. Available columns: {dataset.column_names}")


def load_best_split(dataset_name):
    ds_dict = load_dataset(dataset_name)
    for split in ["validation", "valid", "dev", "test", "train"]:
        if split in ds_dict:
            return ds_dict[split], split
    first_split = list(ds_dict.keys())[0]
    return ds_dict[first_split], first_split


def extract_chunk_by_word_offsets(text, word_offsets, start_word_idx, num_words):
    """
    从 text 里按照 whitespace word offset 抽取一个 chunk。
    这样比 ' '.join(words) 更接近原始文本。
    """
    end_word_idx = min(start_word_idx + num_words, len(word_offsets))

    start_char = word_offsets[start_word_idx][1][0]
    end_char = word_offsets[end_word_idx - 1][1][1]

    chunk_text = text[start_char:end_char]
    actual_words = end_word_idx - start_word_idx

    return chunk_text, actual_words


results = []

for lang, dataset_name in DATASETS.items():
    print(f"\nLoading {lang}: {dataset_name}")

    ds, split_name = load_best_split(dataset_name)
    text_col = get_text_column(ds)

    # 先 shuffle example 顺序；对于 English 这种只有一个超长 example 的情况，后面还会在内部抽 chunks
    ds = ds.shuffle(seed=SEED)

    rng = random.Random(SEED)

    total_words = 0
    total_tokens = 0
    sampled_segments = 0
    source_examples = 0

    for example in ds:
        if total_words >= MAX_WORDS_PER_LANG:
            break

        text = example[text_col]

        if not isinstance(text, str):
            continue

        text = text.strip()
        if not text:
            continue

        # 得到每个 whitespace word 在原始文本中的位置
        word_offsets = whitespace.pre_tokenize_str(text)
        n_words = len(word_offsets)

        if n_words == 0:
            continue

        source_examples += 1

        # 把这个 example 切成很多 chunk 的起点
        chunk_starts = list(range(0, n_words, CHUNK_WORDS))

        # 关键：随机打乱 chunk 起点，不要只取文本开头
        rng.shuffle(chunk_starts)

        for start_idx in chunk_starts:
            if total_words >= MAX_WORDS_PER_LANG:
                break

            remaining_words = MAX_WORDS_PER_LANG - total_words
            words_to_take = min(CHUNK_WORDS, remaining_words, n_words - start_idx)

            if words_to_take <= 0:
                continue

            chunk_text, words_used = extract_chunk_by_word_offsets(
                text=text,
                word_offsets=word_offsets,
                start_word_idx=start_idx,
                num_words=words_to_take,
            )

            tokens_used = count_tokens(chunk_text)

            total_words += words_used
            total_tokens += tokens_used
            sampled_segments += 1

    if total_words < MAX_WORDS_PER_LANG:
        print(
            f"Warning: {lang} only has {total_words} words, "
            f"less than target {MAX_WORDS_PER_LANG}."
        )

    fertility = total_tokens / total_words

    results.append({
        "language": lang,
        "dataset": dataset_name,
        "split": split_name,
        "source_examples": source_examples,
        "sampled_segments": sampled_segments,
        "words": total_words,
        "tokens": total_tokens,
        "fertility": fertility,
        "words_per_1M_tokens": 1_000_000 / fertility,
    })


df = pd.DataFrame(results)

eng_fertility = df.loc[df["language"] == "English", "fertility"].iloc[0]
df["tokenization_tax_vs_English"] = df["fertility"] / eng_fertility - 1

print("\n=== Fertility Results ===")
print(df.to_string(index=False))


Loading English: BabyLM-community/babylm-eng


Using the latest cached version of the dataset since BabyLM-community/babylm-eng couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at /Users/hui/.cache/huggingface/datasets/BabyLM-community___babylm-eng/default/0.0.0/b78a9336aacf2d876aeb6d289869191443f8d41f (last modified on Sat Jun  6 01:31:00 2026).



Loading Dutch: BabyLM-community/babylm-nld


Using the latest cached version of the dataset since BabyLM-community/babylm-nld couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at /Users/hui/.cache/huggingface/datasets/BabyLM-community___babylm-nld/default/0.0.0/1aa063f7e59b84090489bcab6ca1cfa2b911188d (last modified on Fri May 22 22:25:42 2026).



Loading Indonesian: BabyLM-community/babylm-ind


Using the latest cached version of the dataset since BabyLM-community/babylm-ind couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at /Users/hui/.cache/huggingface/datasets/BabyLM-community___babylm-ind/default/0.0.0/b4d8e2dea3a41835396433dbb32f4532fb7a8644 (last modified on Fri May 22 22:25:42 2026).



Loading Javanese: BabyLM-community/babylm-jav


Using the latest cached version of the dataset since BabyLM-community/babylm-jav couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at /Users/hui/.cache/huggingface/datasets/BabyLM-community___babylm-jav/default/0.0.0/be06eee0846e325a926533e37dca3733fc7e1136 (last modified on Fri Jun  5 18:37:28 2026).



=== Fertility Results ===
  language                     dataset split  source_examples  sampled_segments  words  tokens  fertility  words_per_1M_tokens  tokenization_tax_vs_English
   English BabyLM-community/babylm-eng train              255               300 300000  260209   0.867363         1.152919e+06                     0.000000
     Dutch BabyLM-community/babylm-nld train              710               740 300000  348320   1.161067         8.612770e+05                     0.338616
Indonesian BabyLM-community/babylm-ind train              122               142 300000  342638   1.142127         8.755596e+05                     0.316780
  Javanese BabyLM-community/babylm-jav train             1843              1852 300000  476282   1.587607         6.298789e+05                     0.830383


### check the data quality of JAV: padding & child books

In [8]:
from datasets import load_dataset
from tokenizers.pre_tokenizers import Whitespace
import pandas as pd

whitespace = Whitespace()

def get_text_column(dataset):
    candidates = ["text", "sentence", "content", "raw_text"]
    for col in candidates:
        if col in dataset.column_names:
            return col
    raise ValueError(f"Cannot find text column. Available columns: {dataset.column_names}")

ds = load_dataset("BabyLM-community/babylm-jav", split="train")
text_col = get_text_column(ds)

rows = []

for ex in ds:
    text = ex[text_col]
    if not isinstance(text, str):
        continue

    text = text.strip()
    if not text:
        continue

    n_words = len(whitespace.pre_tokenize_str(text))

    row = {"n_words": n_words}

    # 如果 dataset 有 category，比如 child-books / padding，也一起记录
    if "category" in ds.column_names:
        row["category"] = ex["category"]

    rows.append(row)

df_len = pd.DataFrame(rows)

print("Total examples:", len(df_len))
print("Total words:", df_len["n_words"].sum())
print("\nWord length statistics:")
print(df_len["n_words"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))

if "category" in df_len.columns:
    print("\nWords by category:")
    print(df_len.groupby("category")["n_words"].agg(["count", "sum", "mean", "median"]))

Using the latest cached version of the dataset since BabyLM-community/babylm-jav couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at /Users/hui/.cache/huggingface/datasets/BabyLM-community___babylm-jav/default/0.0.0/be06eee0846e325a926533e37dca3733fc7e1136 (last modified on Thu Jul  2 14:11:22 2026).


Total examples: 6657
Total words: 1144557

Word length statistics:
count     6657.000000
mean       171.932853
std        927.290681
min          4.000000
25%        114.000000
50%        122.000000
75%        132.000000
90%        145.000000
95%        160.200000
99%        740.880000
max      36810.000000
Name: n_words, dtype: float64

Words by category:
             count     sum         mean  median
category                                       
child-books    203  397792  1959.566502   390.0
padding       6454  746765   115.705764   121.0


In [9]:
from collections import defaultdict
from datasets import load_dataset
from tokenizers.pre_tokenizers import Whitespace
from tokenizers import Tokenizer
from pathlib import Path
import random

TOKENIZER_PATH = Path("../shared_tokenizer2/tokenizer.json").resolve()
tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))
whitespace = Whitespace()

MAX_WORDS_PER_LANG = 300_000
CHUNK_WORDS = 5_000
SEED = 42

ds = load_dataset("BabyLM-community/babylm-jav", split="train")
ds = ds.shuffle(seed=SEED)

text_col = "text"
rng = random.Random(SEED)

category_stats = defaultdict(lambda: {
    "source_examples": 0,
    "sampled_segments": 0,
    "words": 0,
    "tokens": 0,
})

total_words = 0
total_tokens = 0

for example in ds:
    if total_words >= MAX_WORDS_PER_LANG:
        break

    text = example[text_col]
    if not isinstance(text, str) or not text.strip():
        continue

    text = text.strip()
    word_offsets = whitespace.pre_tokenize_str(text)
    n_words = len(word_offsets)

    if n_words == 0:
        continue

    category = example["category"] if "category" in ds.column_names else "unknown"
    example_used = False

    chunk_starts = list(range(0, n_words, CHUNK_WORDS))
    rng.shuffle(chunk_starts)

    for start_idx in chunk_starts:
        if total_words >= MAX_WORDS_PER_LANG:
            break

        remaining_words = MAX_WORDS_PER_LANG - total_words
        words_to_take = min(CHUNK_WORDS, remaining_words, n_words - start_idx)

        if words_to_take <= 0:
            continue

        end_word_idx = start_idx + words_to_take
        start_char = word_offsets[start_idx][1][0]
        end_char = word_offsets[end_word_idx - 1][1][1]

        chunk_text = text[start_char:end_char]
        tokens_used = len(tokenizer.encode(chunk_text).ids)

        total_words += words_to_take
        total_tokens += tokens_used

        category_stats[category]["sampled_segments"] += 1
        category_stats[category]["words"] += words_to_take
        category_stats[category]["tokens"] += tokens_used

        if not example_used:
            category_stats[category]["source_examples"] += 1
            example_used = True

print("Total sampled words:", total_words)
print("Total sampled tokens:", total_tokens)
print("Fertility:", total_tokens / total_words)

print("\nCategory breakdown:")
for cat, stats in category_stats.items():
    stats["fertility"] = stats["tokens"] / stats["words"]
    stats["word_share"] = stats["words"] / total_words
    print(cat, stats)

Using the latest cached version of the dataset since BabyLM-community/babylm-jav couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at /Users/hui/.cache/huggingface/datasets/BabyLM-community___babylm-jav/default/0.0.0/be06eee0846e325a926533e37dca3733fc7e1136 (last modified on Thu Jul  2 14:11:22 2026).


Total sampled words: 300000
Total sampled tokens: 479986
Fertility: 1.5999533333333333

Category breakdown:
padding {'source_examples': 1787, 'sampled_segments': 1787, 'words': 206246, 'tokens': 309450, 'fertility': 1.500392734889404, 'word_share': 0.6874866666666667}
child-books {'source_examples': 56, 'sampled_segments': 65, 'words': 93754, 'tokens': 170536, 'fertility': 1.8189730571495617, 'word_share': 0.3125133333333333}


## fertility - only child books(JAV)

In [10]:
import random
import pandas as pd
from pathlib import Path
from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.pre_tokenizers import Whitespace

# Load tokenizer
TOKENIZER_PATH = Path("../shared_tokenizer2/tokenizer.json").resolve()
assert TOKENIZER_PATH.exists(), f"Tokenizer not found: {TOKENIZER_PATH}"

tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))
whitespace = Whitespace()

DATASETS = {
    "English": "BabyLM-community/babylm-eng",
    "Dutch": "BabyLM-community/babylm-nld",
    "Indonesian": "BabyLM-community/babylm-ind",
    "Javanese": "BabyLM-community/babylm-jav",
}

MAX_WORDS_PER_LANG = 300_000

# 每个 chunk 的大小
CHUNK_WORDS = 5_000
SEED = 42


def count_tokens(text):
    # 重要：分 chunk 算的时候，不要给每个 chunk 加 special tokens
    return len(tokenizer.encode(text, add_special_tokens=False).ids)


def get_text_column(dataset):
    candidates = ["text", "sentence", "content", "raw_text"]
    for col in candidates:
        if col in dataset.column_names:
            return col
    raise ValueError(f"Cannot find text column. Available columns: {dataset.column_names}")


def load_best_split(dataset_name):
    ds_dict = load_dataset(dataset_name)
    for split in ["validation", "valid", "dev", "test", "train"]:
        if split in ds_dict:
            return ds_dict[split], split
    first_split = list(ds_dict.keys())[0]
    return ds_dict[first_split], first_split


def extract_chunk_by_word_offsets(text, word_offsets, start_word_idx, num_words):
    """
    从 text 里按照 whitespace word offset 抽取一个 chunk。
    这样比 ' '.join(words) 更接近原始文本。
    """
    end_word_idx = min(start_word_idx + num_words, len(word_offsets))

    start_char = word_offsets[start_word_idx][1][0]
    end_char = word_offsets[end_word_idx - 1][1][1]

    chunk_text = text[start_char:end_char]
    actual_words = end_word_idx - start_word_idx

    return chunk_text, actual_words


results = []

for lang, dataset_name in DATASETS.items():
    print(f"\nLoading {lang}: {dataset_name}")

    ds, split_name = load_best_split(dataset_name)
    text_col = get_text_column(ds)

    # Only for Javanese: use child-books subset, excluding padding
    if lang == "Javanese":
        assert "category" in ds.column_names, "Javanese dataset has no category column."
        ds = ds.filter(lambda x: x["category"] == "child-books")
        print("Filtered Javanese to child-books only.")
        print("Remaining examples:", len(ds))

    ds = ds.shuffle(seed=SEED)

    rng = random.Random(SEED)

    total_words = 0
    total_tokens = 0
    sampled_segments = 0
    source_examples = 0

    for example in ds:
        if total_words >= MAX_WORDS_PER_LANG:
            break

        text = example[text_col]

        if not isinstance(text, str):
            continue

        text = text.strip()
        if not text:
            continue

        # 得到每个 whitespace word 在原始文本中的位置
        word_offsets = whitespace.pre_tokenize_str(text)
        n_words = len(word_offsets)

        if n_words == 0:
            continue

        source_examples += 1

        # 把这个 example 切成很多 chunk 的起点
        chunk_starts = list(range(0, n_words, CHUNK_WORDS))

        # 关键：随机打乱 chunk 起点，不要只取文本开头
        rng.shuffle(chunk_starts)

        for start_idx in chunk_starts:
            if total_words >= MAX_WORDS_PER_LANG:
                break

            remaining_words = MAX_WORDS_PER_LANG - total_words
            words_to_take = min(CHUNK_WORDS, remaining_words, n_words - start_idx)

            if words_to_take <= 0:
                continue

            chunk_text, words_used = extract_chunk_by_word_offsets(
                text=text,
                word_offsets=word_offsets,
                start_word_idx=start_idx,
                num_words=words_to_take,
            )

            tokens_used = count_tokens(chunk_text)

            total_words += words_used
            total_tokens += tokens_used
            sampled_segments += 1

    if total_words < MAX_WORDS_PER_LANG:
        print(
            f"Warning: {lang} only has {total_words} words, "
            f"less than target {MAX_WORDS_PER_LANG}."
        )

    fertility = total_tokens / total_words

    results.append({
        "language": lang,
        "dataset": dataset_name,
        "split": split_name,
        "source_examples": source_examples,
        "sampled_segments": sampled_segments,
        "words": total_words,
        "tokens": total_tokens,
        "fertility": fertility,
        "words_per_1M_tokens": 1_000_000 / fertility,
    })


df = pd.DataFrame(results)

eng_fertility = df.loc[df["language"] == "English", "fertility"].iloc[0]
df["tokenization_tax_vs_English"] = df["fertility"] / eng_fertility - 1

print("\n=== Fertility Results ===")
print(df.to_string(index=False))


Loading English: BabyLM-community/babylm-eng


Using the latest cached version of the dataset since BabyLM-community/babylm-eng couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at /Users/hui/.cache/huggingface/datasets/BabyLM-community___babylm-eng/default/0.0.0/b78a9336aacf2d876aeb6d289869191443f8d41f (last modified on Sat Jun  6 01:31:00 2026).



Loading Dutch: BabyLM-community/babylm-nld


Using the latest cached version of the dataset since BabyLM-community/babylm-nld couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at /Users/hui/.cache/huggingface/datasets/BabyLM-community___babylm-nld/default/0.0.0/1aa063f7e59b84090489bcab6ca1cfa2b911188d (last modified on Fri May 22 22:25:42 2026).



Loading Indonesian: BabyLM-community/babylm-ind


Using the latest cached version of the dataset since BabyLM-community/babylm-ind couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at /Users/hui/.cache/huggingface/datasets/BabyLM-community___babylm-ind/default/0.0.0/b4d8e2dea3a41835396433dbb32f4532fb7a8644 (last modified on Fri May 22 22:25:42 2026).



Loading Javanese: BabyLM-community/babylm-jav


Using the latest cached version of the dataset since BabyLM-community/babylm-jav couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at /Users/hui/.cache/huggingface/datasets/BabyLM-community___babylm-jav/default/0.0.0/be06eee0846e325a926533e37dca3733fc7e1136 (last modified on Thu Jul  2 14:11:22 2026).


Filter:   0%|          | 0/6657 [00:00<?, ? examples/s]

Filtered Javanese to child-books only.
Remaining examples: 203

=== Fertility Results ===
  language                     dataset split  source_examples  sampled_segments  words  tokens  fertility  words_per_1M_tokens  tokenization_tax_vs_English
   English BabyLM-community/babylm-eng train              255               300 300000  260209   0.867363         1.152919e+06                     0.000000
     Dutch BabyLM-community/babylm-nld train              710               740 300000  348320   1.161067         8.612770e+05                     0.338616
Indonesian BabyLM-community/babylm-ind train              122               142 300000  342638   1.142127         8.755596e+05                     0.316780
  Javanese BabyLM-community/babylm-jav train              152               188 300000  538958   1.796527         5.566296e+05                     1.071250
